# Discrete example module
This notebook demonstrates using the discrete example module.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax.numpy as jnp
from popsim.modules.discrete_example import DiscreteExample, ExampleState
from popsim.interp import resolve_paths
from popsim.tree_util import leaves_as_array
from popsim.param_utils import build_vectorized_params, make_time_base
import jax
import typing

state = DiscreteExample.State(example_state=ExampleState.Rotating)


t0=0.0
t1=10.0
params1 = DiscreteExample.Params(time={t0:t0, t1:t1}, time_to_decel=0.1, time_to_lock=0.3)
dt=0.01
time_base = make_time_base(t0=t0, t1=t1, dt=dt)
interp_type='linear'
from copy import deepcopy

params2 = deepcopy(params1) # Make two sets of params to test that the vec_step function works properly
params2.time_to_decel=0.4
params2.time_to_lock=0.5
params = [params1, params2]

#TODO: Move to popsim.simulate
if not isinstance(params, typing.Sequence):
    params = [params]
params_vectorized, multi_sim = build_vectorized_params(params, time_base, interp_type)
params_axes = jax.tree_map(lambda x: 0, params_vectorized)
discrete_module = DiscreteExample(config=DiscreteExample.Config())

#sol = simulate_discrete(discrete_module, times, state, params)

In [ ]:
#TODO: Move these to popsim.simulate

import equinox as eqx
@eqx.filter_jit
def vec_step(model, t0, dt, state0, params_vectorized):
    params_axes = jax.tree_map(lambda x: 0, params_vectorized)

    # Perform a vectorized simulation.
    state_new, out, t = jax.vmap(
        discrete_step,
        in_axes=(None, None, None, None, params_axes),
    )(model, t0, dt, state0, params_vectorized)
    return state_new, out, t

@eqx.filter_jit
def discrete_step(model,t0,dt,state0,params):
    params_resolved = resolve_paths(params, t0)
    t=t0+dt
    state_new, out = model(state, params_resolved)
    return state_new, out, t

vec_step_jitted = jax.jit(vec_step)

In [ ]:
def sim(state):
    t=0
    states = []
    for i in jnp.arange(len(time_base)):
        state, out, t = vec_step_jitted(discrete_module, t, dt, state, params_vectorized)
        t=t[0]
        states.append(state)
    return states

%time states = sim(state)

In [ ]:
example_state = leaves_as_array(states)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,1)
ax.plot(time_base,example_state[:len(time_base)])
ax.set_yticks([e.value for e in ExampleState])
ax.set_yticklabels([e.name for e in ExampleState])
ax.set_ylabel('Time [s]')
